## K-Nearest Neighbors (KNN)

KNN is a supervised, non-parametric, instance-based (lazy) algorithm used mainly for **classification** and also for **regression**.

1. Compute distance from the new point to all training points.
2. Select the **k** nearest neighbors.
3. Predict:
   - Classification: majority class
   - Regression: average target value

### Choosing `k`

- Small `k`: sensitive to noise (can overfit)
- Large `k`: smoother but may underfit
- Best practice: choose `k` using cross-validation (or an elbow/error curve)
- In binary classification, use an odd `k` to reduce ties

### Distance Metrics in KNN

KNN relies on measuring **how close** points are. Different metrics suit different data shapes.

---

**1. Euclidean Distance** — straight-line ("as the crow flies")

$$d(\mathbf{x}, \mathbf{y}) = \sqrt{\sum_{i=1}^{n}(x_i - y_i)^2}$$

Works well when **all features are on similar scales** and contribute equally.

---

**2. Manhattan Distance** — grid/taxicab path (sum of absolute differences)

$$d(\mathbf{x}, \mathbf{y}) = \sum_{i=1}^{n}|x_i - y_i|$$

Better when features are **independent or on different scales**, or data lies on a grid-like structure. Less sensitive to large outliers in a single dimension because it doesn't square differences.

---

**3. Minkowski Distance** — generalized family

$$d(\mathbf{x}, \mathbf{y}) = \left(\sum_{i=1}^{n}|x_i - y_i|^p\right)^{\frac{1}{p}}$$

- $p=1$ → Manhattan
- $p=2$ → Euclidean
- Higher $p$ → emphasizes the **largest** single-feature difference

Lets you **tune** how much weight big vs. small per-feature gaps get.

---

**Why not always Euclidean?**

| Issue | Explanation |
|---|---|
| **High-dimensional data** | Euclidean distances converge (curse of dimensionality); Manhattan often discriminates better. |
| **Outlier sensitivity** | Squaring amplifies large gaps — Manhattan is more robust. |
| **Mixed feature scales** | Manhattan treats each axis independently, reducing distortion from scale mismatches. |

**Rule of thumb:** start with Euclidean; switch to Manhattan for high-dim / noisy data; use Minkowski to experiment with $p$.

### How KNN Works

1. **Choose K** — pick the number of neighbors to consider.
2. **Compute distances** — measure distance (e.g. Euclidean) from the new point to every training point.
3. **Find K nearest neighbors** — select the K points with the smallest distances.
4. **Predict:**
   - *Classification* → majority vote among the K neighbors.
   - *Regression* → average of the K neighbors' values.

## <br>Implementing KNN from Scratch

#### Code Breakdown

| Expression | Meaning | Example |
|---|---|---|
| `key=lambda x: x[0]` | Sort tuples by distance | `[(3.5,'B'),(1.0,'A')]` → `[(1.0,'A'),(3.5,'B')]` |
| `distances[:k]` | Take k closest | k=3 → first 3 sorted tuples |
| `[label for _, label in ...]` | Extract labels (`_` = ignore dist) | `[(1.0,'A'),(2.0,'A'),(3.5,'B')]` → `['A','A','B']` |
| `Counter(labels)` | Count each label | `Counter({'A':2, 'B':1})` |
| `.most_common(1)[0][0]` | Get top label | `[('A',2)]` → `('A',2)` → `'A'` |

In [ ]:
import numpy as np
from collections import Counter

def euclidean_distance(point1, point2):
    return np.sqrt(np.sum(np.array(point1) - np.array(point2))**2)

def mathatton_distance(point1, point2):
    return np.sqrt(np.sum(np.array(point1) - np.array(point2)))

def knn_predict(training_data, training_labels, test_point, k):
    distances = []

    for i in range(len(training_data)):
        dist = euclidean_distance(test_point, training_data[i])
        distances.append((dist, training_labels[i]))
    
    distances.sort(key=lambda x: x[0])
    k_nearest_labels = [label for _, label in distances[:k]]

    return Counter(k_nearest_labels).most_common(1)[0][0]


training_data = [[1, 2], [2, 3], [3, 4], [6, 7], [7, 8]]
training_labels = ['A', 'A', 'A', 'B', 'B']
test_point = [4, 5]
k = 3


prediction = knn_predict(training_data, training_labels, test_point, k)
print(prediction)


['A', 'A', 'B']
A


## KNN with Scikit-Learn

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load dataset
iris = load_iris()
X, y = iris.data, iris.target

# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features (important for distance-based algorithms)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Train KNN
knn = KNeighborsClassifier(n_neighbors=5, metric='euclidean')
knn.fit(X_train, y_train)

# Predict & evaluate
y_pred = knn.predict(X_test)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}\n")
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Accuracy: 1.00

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       1.00      1.00      1.00         9
   virginica       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



## Applications

- **Recommendation systems** — find users with similar preferences to suggest movies, products, etc.
- **Spam detection** — classify emails by comparing to known spam/non-spam examples.
- **Customer segmentation** — group customers by similar shopping behavior.
- **Speech recognition** — match spoken words to known patterns.

## Advantages & Disadvantages

| Advantages | Disadvantages |
|---|---|
| Simple to understand and implement | Slow on large datasets (compares every point) |
| No training step — just stores data | Struggles with high-dimensional data (curse of dimensionality) |
| Few hyperparameters (k, distance metric) | Sensitive to noise and irrelevant features |
| Works for both classification and regression | Can overfit with small k or unclean data |